# Copyright (c) 2026 hidemi-k / Licensed under the MIT License

# Netmiko × Agent Framework v2

## 🌐 ネットワーク機器分析エージェント（agent-framework対応）

このノートブックは `netmiko_agent_framework` に以下の改修を適用したものです。

### v2 改修内容

| 改修 | 優先度 | 効果 |
|------|--------|------|
| **① GroqChatClient廃止 → OpenAIChatClient (MAFネイティブ)** | 🎯 主目的 | `_inner_get_response` 手動実装が不要に。MAF全機能を享受 |
| **② 状態管理の簡素化** | 低 | グローバル変数廃止・Native型変換処理を削除 |
| **③ 暴走防止 (max_calls)** | 低 | エージェントAPIコール上限によるネットワーク事故防止 |
| **④ Self-Correction ループ** | 中 | Reviewer→Generator フィードバックで分析精度向上 |

### OpenAIChatClient 採用の背景
- GroqはOpenAI互換APIのため `base_url` 変更のみで流用可能
- `BaseChatClient` の手動実装（`_inner_get_response`・型変換）が不要になる
- ログ分析用途のため設定変更を伴わず、MAFが提供する会話履歴管理・エラーハンドリングをそのまま活用

## 📦 必要なライブラリのインストール

```bash
pip install agent-framework netmiko openai
```

In [1]:
import asyncio
import os
import configparser
from typing import List, Dict, Any
from datetime import datetime

# Netmiko
from netmiko import ConnectHandler
from netmiko.exceptions import AuthenticationException, NetMikoTimeoutException, SSHException

# agent-framework rc3
# ★ OpenAIChatClient は agent_framework.openai サブモジュールからインポート
#   （agent_framework トップレベルには存在しない）
from agent_framework import (
    Agent,
    Message
)
from agent_framework.openai import OpenAIChatClient  # ★ 正しいimportパス

print("✅ インポート完了")
print("   OpenAIChatClient: agent_framework.openai から正しくインポート")

✅ インポート完了
   OpenAIChatClient: agent_framework.openai から正しくインポート


## 🔑 APIキー読み込み

キー変数名は `GROQ_API_KEY` を維持します（config.ini の構造を変更しないため）。
`OpenAIChatClient` の `api_key` 引数に渡す際に使用します。

In [2]:
# 設定ファイルからAPIキーを読み込み（現状維持）
config = configparser.ConfigParser()
config_paths = [
    '/home/admin/config/config.ini',
    './config.ini',
    os.path.expanduser('~/config/config.ini')
]

GROQ_API_KEY = None
for path in config_paths:
    if os.path.exists(path):
        config.read(path)
        if 'GROQ' in config and 'GROQ_API_KEY' in config['GROQ']:
            GROQ_API_KEY = config['GROQ']['GROQ_API_KEY']
            print(f"✅ APIキーを {path} から読み込みました")
            break

if not GROQ_API_KEY:
    GROQ_API_KEY = os.getenv('GROQ_API_KEY')
    if GROQ_API_KEY:
        print("✅ APIキーを環境変数から読み込みました")
    else:
        print("⚠️  APIキーが見つかりません - 手動で設定してください")
        # GROQ_API_KEY = "your-api-key-here"

print(f"✅ APIキー設定完了 (OpenAIChatClient の api_key 引数に渡します)")

✅ APIキーを /home/admin/config/config.ini から読み込みました
✅ APIキー設定完了 (OpenAIChatClient の api_key 引数に渡します)


## 🤖 OpenAIChatClient ファクトリ関数

### v1 との比較

```python
# ── v1: GroqChatClient (BaseChatClient 手動実装) ──────────────────────
class GroqChatClient(BaseChatClient):
    async def _inner_get_response(self, messages, ...):
        # ① role の .value 取得など Native型変換を手動処理
        # ② グローバル変数 _conversation_history に手動で追記
        # ③ ChatResponse.from_dict(...) を手動で組み立て
        ...

# ── v2: OpenAIChatClient (MAFネイティブ) ─────────────────────────────
def make_client(agent_name):
    return OpenAIChatClient(
        api_key  = GROQ_API_KEY,
        model    = "llama-3.3-70b-versatile",
        base_url = "https://api.groq.com/openai/v1",  # Groq互換エンドポイント
    )
# → 型変換・履歴管理・ChatResponse組み立ては MAF が全て処理
```

### モデル指定
- `llama-3.3-70b-versatile`（推奨・デフォルト）
- `llama-3.1-8b-instant`（高速）

In [3]:
# ★ v2 改修①: GroqChatClient クラス定義を削除
#              OpenAIChatClient をそのまま使うファクトリ関数に置き換え
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
DEFAULT_MODEL = "llama-3.3-70b-versatile"

def make_client(model_id: str = DEFAULT_MODEL) -> OpenAIChatClient:
    """
    MAFネイティブ OpenAIChatClient を生成する。

    Groq は OpenAI 互換 API のため base_url を差し替えるだけで流用可能。
    BaseChatClient の手動実装（_inner_get_response・型変換）は一切不要。

    引数名は MAF 公式ドキュメント準拠:
      model_id  : モデル名（'model' ではなく 'model_id'）
      api_key   : APIキー
      base_url  : Groq互換エンドポイント
    """
    return OpenAIChatClient(
        model_id=model_id,       # ★ 'model' ではなく 'model_id'
        api_key=GROQ_API_KEY,
        base_url=GROQ_BASE_URL,
    )

print("✅ make_client() 定義完了")
print(f"   エンドポイント  : {GROQ_BASE_URL}")
print(f"   デフォルトモデル: {DEFAULT_MODEL}")

✅ make_client() 定義完了
   エンドポイント  : https://api.groq.com/openai/v1
   デフォルトモデル: llama-3.3-70b-versatile


## 🔌 Netmiko接続とコマンド実行

In [4]:
def run_netmiko_commands(
    ip: str,
    port: str,
    username: str,
    password: str,
    device_type: str,
    commands: List[str]
) -> List[tuple]:
    """
    Netmikoでネットワーク機器に接続してコマンドを実行

    Returns:
        List[tuple]: [(command, output), ...]
    """
    try:
        device = {
            "device_type": device_type,
            "port": port,
            "ip": ip,
            "username": username,
            "password": password,
        }

        print(f"🔌 {ip} に接続中...")
        connection = ConnectHandler(**device)

        results = []
        for command in commands:
            print(f"  📝 実行中: {command}")
            output = connection.send_command(command, read_timeout=120)
            results.append((command, output))

        connection.disconnect()
        print(f"✅ {len(commands)} 個のコマンドを実行完了")
        return results

    except AuthenticationException:
        return [("認証エラー", "ユーザー名またはパスワードが正しくありません。")]
    except NetMikoTimeoutException:
        return [("接続タイムアウト", "指定されたIPアドレスまたはポートへの接続に失敗しました。")]
    except SSHException as e:
        return [("SSHエラー", f"SSH接続中にエラーが発生しました: {str(e)}")]
    except Exception as e:
        return [("予期せぬエラー", f"Netmikoの実行中にエラーが発生しました: {type(e).__name__}: {str(e)}")]

print("✅ Netmiko関数定義完了")

✅ Netmiko関数定義完了


## 🤖 エージェント設定とラウンドロビン実行

### v2 改修まとめ

| 改修 | v1 | v2 |
|------|----|----|---|
| ① クライアント | `GroqChatClient(BaseChatClient)` 手動実装 | `OpenAIChatClient` MAFネイティブ |
| ② 状態管理 | グローバル変数 `_conversation_history` + Native型変換 | `self.conversation_history: List[Message]` インスタンス変数 |
| ③ 暴走防止 | なし | `max_calls` でAPIコール上限を設定 |
| ④ Self-Correction | なし（一方通行） | Reviewer → Generator フィードバックループ |

In [5]:
class NetworkAnalysisTeam:
    """
    ネットワーク分析チーム - ラウンドロビン方式
    agent-framework rc3 + v2 改修適用版

    v2 改修:
      ① GroqChatClient 廃止 → OpenAIChatClient (MAFネイティブ)
      ② 状態管理の簡素化: グローバル変数廃止・Native型変換処理を削除
      ③ 暴走防止: max_calls でAPIコール総数に上限
      ④ Self-Correction: Reviewer エージェントによるフィードバックループ
    """

    def __init__(
        self,
        enabled_agents: List[str],
        custom_message: str = "",
        max_calls: int = 20,          # ★ 改修③: APIコール上限（暴走防止）
        enable_self_correction: bool = True,  # ★ 改修④: Self-Correctionの有効/無効
        model: str = DEFAULT_MODEL,
    ):
        self.max_calls = max_calls
        self.enable_self_correction = enable_self_correction
        self.model = model
        self._call_count = 0           # ★ 改修③: コール数カウンター

        # ★ 改修②: グローバル変数廃止 → インスタンス変数で会話履歴を管理
        self.conversation_history: List[Message] = []

        self.agents = []
        self.agent_configs = {
            'Engineer1': {
                'name': '設計担当',
                'emoji': '🔧',
                'instructions': 'あなたはネットワーク設計担当者です。技術的な観点から分析を行い、日本語で2-3文で簡潔に述べてください。'
            },
            'Engineer2': {
                'name': '運用担当',
                'emoji': '📋',
                'instructions': 'あなたはネットワーク運用担当者です。運用上の観点から要点を整理し、日本語で2-3文で簡潔に述べてください。'
            },
            'Translator': {
                'name': '翻訳担当',
                'emoji': '🌐',
                'instructions': 'あなたは技術翻訳担当です。英語の内容を正確かつ自然な日本語に翻訳し、必要に応じて技術的な文脈を補足してください。日本語で2-3文で述べてください。'
            },
            'CustomAgent': {
                'name': 'カスタム',
                'emoji': '🤖',
                'instructions': custom_message or 'あなたはカスタムエージェントです。指定されたタスクを実行してください。'
            },
            # ★ 改修④: Self-Correction 用 Reviewer エージェント定義
            '_Reviewer': {
                'name': 'レビュー担当',
                'emoji': '👁️',
                'instructions': (
                    'あなたはネットワーク分析のレビュー担当です。'
                    '複数のエンジニアによる分析内容を確認し、'
                    '見落とし・矛盾・技術的な誤りがあれば指摘してください。'
                    '問題がなければ「分析は適切です」と述べてください。'
                    '日本語で2-3文で回答してください。'
                )
            }
        }

        self._initialize_agents(enabled_agents, custom_message)

    def _initialize_agents(self, enabled_agents: List[str], custom_message: str):
        """エージェントの初期化"""
        print("\n🔧 エージェント初期化中...")

        for agent_id in enabled_agents:
            if agent_id not in self.agent_configs:
                continue

            config = self.agent_configs[agent_id]
            instructions = config['instructions']
            if agent_id == 'CustomAgent' and custom_message:
                instructions = custom_message

            # ★ 改修①: GroqChatClient() → make_client() (OpenAIChatClient)
            agent = Agent(
                name=config['name'],
                client=make_client(self.model),
                instructions=instructions
            )

            self.agents.append({
                'id': agent_id,
                'agent': agent,
                'config': config
            })
            print(f"  ✓ {config['emoji']} {config['name']}")

        # ★ 改修④: Self-Correction 用 Reviewer を追加
        if self.enable_self_correction:
            rev_config = self.agent_configs['_Reviewer']
            self.reviewer = Agent(
                name=rev_config['name'],
                client=make_client(self.model),
                instructions=rev_config['instructions']
            )
            print(f"  ✓ {rev_config['emoji']} {rev_config['name']} (Self-Correction用)")
        else:
            self.reviewer = None

        print(f"✅ {len(self.agents)} 人のエージェント準備完了")
        print(f"   暴走防止: max_calls={self.max_calls}")
        print(f"   Self-Correction: {'有効' if self.enable_self_correction else '無効'}\n")

    def _extract_response_text(self, response) -> str:
        """レスポンスからテキストを抽出（rc3対応）"""
        if hasattr(response, 'text'):
            return str(response.text)
        if hasattr(response, 'messages') and response.messages:
            for msg in response.messages:
                if hasattr(msg, 'text'):
                    return str(msg.text)
        return str(response)

    def _check_call_limit(self) -> bool:
        """
        ★ 改修③: 暴走防止チェック
        max_calls を超えたら False を返し、処理を中断する。
        ネットワーク機器への誤操作を引き起こす連鎖呼び出しを防ぐ。
        """
        self._call_count += 1
        if self._call_count > self.max_calls:
            print(f"⛔ [暴走防止] APIコール数が上限({self.max_calls})を超えました。処理を中断します。")
            return False
        return True

    async def analyze_command_output(
        self,
        command: str,
        output: str,
        user_task: str = "",
        max_turns: int = 1
    ):
        """
        コマンド出力を分析（ラウンドロビン + Self-Correction）

        フロー:
          [Turn 1..N] 各エージェントが順番に分析
               ↓
          [Self-Correction] Reviewer が全分析を確認・指摘
               ↓ (指摘があった場合)
          [Correction] 最初のエージェントが修正分析を出力

        Args:
            command  : 実行したコマンド
            output   : コマンドの出力
            user_task: ユーザーからの追加指示
            max_turns: 各エージェントの発言回数
        """
        print("\n" + "="*70)
        print(f"📊 コマンド: {command}")
        print("="*70)

        # ★ 改修②: グローバル変数ではなくローカルの conversation_history を使用
        #           Message オブジェクト(MAF Native型)のまま扱い、str変換は表示時のみ
        conversation_history = []  # [{speaker, message, turn}, ...]

        initial_task = (
            f"すべての出力は日本語で行ってください。\n"
            f"{user_task}\n\n"
            f"以下の情報を参考にしてください\n\n---\n\n"
            f"ネットワーク機器のコマンド: '{command}'\n\n"
            f"コマンド出力:\n{output}"
        )

        # ── ラウンドロビン分析 ────────────────────────────────────────
        for turn in range(max_turns):
            if max_turns > 1:
                print(f"\n🔄 ターン {turn + 1}/{max_turns}")
                print("-" * 70)

            for agent_info in self.agents:
                agent  = agent_info['agent']
                config = agent_info['config']

                # ★ 改修③: APIコール前に上限チェック
                if not self._check_call_limit():
                    return conversation_history

                # コンテキスト構築
                if turn == 0 and agent_info == self.agents[0]:
                    context = initial_task
                else:
                    history_text = "\n【これまでの分析】\n"
                    for entry in conversation_history:
                        history_text += f"{entry['speaker']}: {entry['message']}\n"
                    context = f"{history_text}\n\n上記を踏まえて、あなたの視点から日本語で分析してください。"

                print(f"\n  {config['emoji']} {config['name']} 分析中...", end=" ")

                try:
                    response = await agent.run(context)
                    result   = self._extract_response_text(response)

                    if result:
                        conversation_history.append({
                            'turn':    turn + 1,
                            'speaker': f"{config['emoji']} {config['name']}",
                            'message': result
                        })
                        print("✓")
                        print(f"  💬 {result}")
                    else:
                        print("⚠️  応答が空")

                except Exception as e:
                    print(f"⚠️  エラー: {e}")

        # ── Self-Correction ───────────────────────────────────────────
        # ★ 改修④: Reviewer がラウンドロビン後の全分析を確認し、誤りがあれば修正
        if self.enable_self_correction and self.reviewer and conversation_history:

            if not self._check_call_limit():
                return conversation_history

            print("\n" + "-"*70)
            print("  👁️  [Self-Correction] Reviewer が分析を確認中...", end=" ")

            summary = "\n".join(
                f"{e['speaker']}: {e['message']}" for e in conversation_history
            )
            review_prompt = (
                f"以下はネットワークコマンド '{command}' の分析結果です。\n\n"
                f"{summary}\n\n"
                f"技術的な誤りや見落としがあれば指摘してください。"
                f"問題なければ「分析は適切です」と述べてください。"
            )

            try:
                review_response = await self.reviewer.run(review_prompt)
                review_text     = self._extract_response_text(review_response)
                print("✓")
                print(f"  📋 {review_text}")

                conversation_history.append({
                    'turn':    'review',
                    'speaker': '👁️  レビュー担当',
                    'message': review_text
                })

                # Reviewer が問題を指摘した場合 → 最初のエージェントが修正
                if "適切" not in review_text and "問題" not in review_text:
                    if not self._check_call_limit():
                        return conversation_history

                    first_agent  = self.agents[0]
                    first_config = first_agent['config']
                    print(f"\n  {first_config['emoji']} [修正] {first_config['name']} が再分析中...", end=" ")

                    correction_prompt = (
                        f"レビュー担当から以下の指摘がありました:\n{review_text}\n\n"
                        f"元の分析: {conversation_history[0]['message']}\n\n"
                        f"指摘を踏まえて修正した分析を日本語で2-3文で述べてください。"
                    )
                    correction_response = await first_agent['agent'].run(correction_prompt)
                    correction_text     = self._extract_response_text(correction_response)

                    if correction_text:
                        print("✓")
                        print(f"  💬 [修正後] {correction_text}")
                        conversation_history.append({
                            'turn':    'correction',
                            'speaker': f"{first_config['emoji']} {first_config['name']} [修正]",
                            'message': correction_text
                        })
                    else:
                        print("⚠️  修正応答が空")

            except Exception as e:
                print(f"⚠️  Reviewer エラー: {e}")

        print(f"\n  📈 APIコール数: {self._call_count} / {self.max_calls}")
        print("\n" + "="*70)
        return conversation_history

print("✅ NetworkAnalysisTeam (v2) 定義完了")

✅ NetworkAnalysisTeam (v2) 定義完了


## 📝 使い方のヒント

### max_calls の目安

```
max_calls ≧ (エージェント数) × MAX_TURNS × (コマンド数) + (Reviewer×コマンド数) + (修正×コマンド数)

例: Engineer×2, MAX_TURNS=1, コマンド×3, Self-Correction有効
    = 2×1×3 + 1×3 + 1×3 = 6 + 3 + 3 = 12 → MAX_CALLS=20 で余裕あり
```

### エージェントの組み合わせ例

1. **基本構成**（2エージェント + Self-Correction）
   ```python
   ENABLED_AGENTS = ['Engineer1', 'Engineer2']
   MAX_TURNS = 1
   ENABLE_SELF_CORRECTION = True
   MAX_CALLS = 20
   ```

2. **翻訳付き**
   ```python
   ENABLED_AGENTS = ['Engineer1', 'Translator']
   MAX_TURNS = 1
   ```

3. **安全優先**（Self-Correction無効・コール数厳格制限）
   ```python
   ENABLED_AGENTS = ['Engineer1', 'Engineer2']
   MAX_TURNS = 1
   ENABLE_SELF_CORRECTION = False
   MAX_CALLS = 10
   ```

4. **深い分析**（複数ターン）
   ```python
   ENABLED_AGENTS = ['Engineer1', 'Engineer2']
   MAX_TURNS = 2
   MAX_CALLS = 30
   ```

### 対応機器タイプ

- `juniper_junos` - Juniper JunOS
- `cisco_ios` - Cisco IOS
- `cisco_nxos` - Cisco NX-OS
- `arista_eos` - Arista EOS
- `nokia_sros` - Nokia SR OS
- その他、Netmiko対応機器

### v1 → v2 変更点まとめ

| 項目 | v1 | v2 |
|------|----|----|---|
| クライアント | `GroqChatClient(BaseChatClient)` 手動実装 | `OpenAIChatClient` MAFネイティブ |
| 状態管理 | グローバル変数 `_conversation_history` | `self.conversation_history` インスタンス変数 |
| Native型変換 | 手動（role.value, contents等） | MAFが処理（不要） |
| 暴走防止 | なし | `max_calls` パラメータ |
| Self-Correction | なし | Reviewer → Generator フィードバック |
| APIキー変数名 | `GROQ_API_KEY` | `GROQ_API_KEY`（変更なし） |
| エンドポイント | `https://api.groq.com/openai/v1` | 同左（`GROQ_BASE_URL`定数化） |

## 🚀 実行例

### 📝 設定

以下のパラメータを環境に合わせて変更してください：

In [6]:
# ネットワーク機器の接続情報
DEVICE_CONFIG = {
    'ip': '192.0.2.1'  # Replace with your device IP,
    'port':        '22',
    'username':    'admin',
    'password': 'YOUR_PASSWORD_HERE',
    'device_type': 'juniper_junos'
}

# 実行するコマンド
COMMANDS = [
    'show version',
    'show bgp summary',
    'show interfaces brief',
    # 'show interfaces extensive et-0/0/0'
]

# 有効にするエージェント
ENABLED_AGENTS = [
    'Engineer1',   # 設計担当
    'Engineer2',   # 運用担当
    # 'Translator',  # 翻訳担当
    # 'CustomAgent'  # カスタムエージェント
]

# カスタムエージェントのメッセージ（CustomAgent を有効にする場合）
CUSTOM_AGENT_MESSAGE = "あなたはセキュリティ専門家です。セキュリティリスクの観点から分析してください。"

# AIへの分析指示
USER_TASK = "ネットワークの状態を分析し、問題点があれば指摘してください。"

# 各エージェントの発言回数
MAX_TURNS = 1  # 2以上にすると複数ターン

# ★ v2: 暴走防止・Self-Correction の設定
MAX_CALLS           = 20     # APIコール上限（エージェント数 × MAX_TURNS × コマンド数 を目安に設定）
ENABLE_SELF_CORRECTION = True  # False にすると Reviewer をスキップ

print("✅ 設定完了")
print(f"   max_calls={MAX_CALLS}, self_correction={ENABLE_SELF_CORRECTION}")

✅ 設定完了
   max_calls=20, self_correction=True


### ステップ1: Netmikoでコマンド実行

In [7]:
print("\n" + "="*70)
print("🔌 Netmikoでコマンド実行")
print("="*70)

netmiko_results = run_netmiko_commands(
    ip=DEVICE_CONFIG['ip'],
    port=DEVICE_CONFIG['port'],
    username=DEVICE_CONFIG['username'],
    password=DEVICE_CONFIG['password'],
    device_type=DEVICE_CONFIG['device_type'],
    commands=COMMANDS
)

print("\n📄 実行結果:")
print("="*70)
for cmd, output in netmiko_results:
    print(f"\n📝 コマンド: {cmd}")
    print(f"出力:\n{output[:500]}...")  # 最初の500文字のみ表示
    print("-" * 70)


🔌 Netmikoでコマンド実行
🔌 172.20.100.21 に接続中...
  📝 実行中: show version
  📝 実行中: show bgp summary
  📝 実行中: show interfaces brief
✅ 3 個のコマンドを実行完了

📄 実行結果:

📝 コマンド: show version
出力:
Hostname: cjunosevo-pe-1
Model: ptx10001-36mr
Junos: 25.4R1.13-EVO
Yocto: 4.0.22
Linux Kernel: 5.15.164-10.22.33.18-yocto-standard-juniper-17155-g5adff1d6d404
JUNOS-EVO OS 64-bit [junos-evo-install-ptx-fixed-x86-64-25.4R1.13-EVO]
...
----------------------------------------------------------------------

📝 コマンド: show bgp summary
出力:


Threading mode: BGP I/O
Default eBGP mode: advertise - accept, receive - accept
Groups: 1 Peers: 1 Down peers: 1
Table          Tot Paths  Act Paths Suppressed    History Damp State    Pending
bgp.evpn.0           
                       0          0          0          0          0          0
Peer                     AS      InPkt     OutPkt    OutQ   Flaps Last Up/Dwn State|#Active/Received/Accepted/Damped...
10.99.2.1             65000   ...
------------------------------------------

### ステップ2: AIエージェントによる分析

In [8]:
# ★ v2: チーム初期化（OpenAIChatClient使用・max_calls・Self-Correction設定）
team = NetworkAnalysisTeam(
    enabled_agents=ENABLED_AGENTS,
    custom_message=CUSTOM_AGENT_MESSAGE,
    max_calls=MAX_CALLS,
    enable_self_correction=ENABLE_SELF_CORRECTION,
    model=DEFAULT_MODEL,
)

# 各コマンド出力を分析
all_analyses = []

for cmd, output in netmiko_results:
    analysis = await team.analyze_command_output(
        command=cmd,
        output=output,
        user_task=USER_TASK,
        max_turns=MAX_TURNS
    )
    all_analyses.append({
        'command':  cmd,
        'analysis': analysis
    })

print("\n✅ 全ての分析が完了しました")
print(f"   総APIコール数: {team._call_count} / {team.max_calls}")


🔧 エージェント初期化中...
  ✓ 🔧 設計担当
  ✓ 📋 運用担当
  ✓ 👁️ レビュー担当 (Self-Correction用)
✅ 2 人のエージェント準備完了
   暴走防止: max_calls=20
   Self-Correction: 有効


📊 コマンド: show version

  🔧 設計担当 分析中... ✓
  💬 ネットワーク機器のバージョン情報から、Junosのバージョンは25.4R1.13-EVOであり、最新のアップデートが適用されていない可能性があります。ネットワークのセキュリティとパフォーマンスを確保するために、最新のアップデートの適用を検討する必要があります。さらに、システムのログを確認してエラーの有無を確認するべきです。

  📋 運用担当 分析中... ✓
  💬 ネットワーク運用担当者としての観点では、ネットワーク機器のバージョンが最新でない可能性はセキュリティ上のリスクとなり得るため、早期のアップデートを推奨します。さらに、システムのログを定期的に確認してエラーを迅速に発見し、トラブルシューティングを行うことが重要です。エラーの早期発見と対応により、ネットワークの安定性とパフォーマンスを維持することができるため、これらの対策を実施する必要があります。

----------------------------------------------------------------------
  👁️  [Self-Correction] Reviewer が分析を確認中... ✓
  📋 分析は適切です。ネットワーク機器のバージョン情報からセキュリティリスクの可能性とアップデートの必要性を指摘し、さらにシステムのログを確認してエラーを早期に発見することで、ネットワークの安定性とパフォーマンスを維持することができるという点は妥当です。設計担当と運用担当それぞれの視点から問題が分析されており、適切な対策が提案されているため、技術的な誤りや見落としは見られません。

  📈 APIコール数: 3 / 20


📊 コマンド: show bgp summary

  🔧 設計担当 分析中... ✓
  💬 ネットワークの状態を分析すると、BGP（ボーダーゲートウェイ・プロトコル）ライセンスキーが欠けているという

### 📊 分析結果のサマリー

In [9]:
print("\n" + "="*70)
print("📊 分析結果サマリー")
print("="*70)

for item in all_analyses:
    print(f"\n📝 コマンド: {item['command']}")
    print("-" * 70)

    for entry in item['analysis']:
        # ★ v2: turn種別（通常/review/correction）でラベルを変える
        turn_label = {
            'review':     '🔍 [レビュー]',
            'correction': '🔧 [修正]'
        }.get(str(entry.get('turn', '')), f"T{entry.get('turn','')}")

        print(f"  {turn_label} {entry['speaker']}:")
        print(f"    {entry['message']}")
        print()

print("="*70)


📊 分析結果サマリー

📝 コマンド: show version
----------------------------------------------------------------------
  T1 🔧 設計担当:
    ネットワーク機器のバージョン情報から、Junosのバージョンは25.4R1.13-EVOであり、最新のアップデートが適用されていない可能性があります。ネットワークのセキュリティとパフォーマンスを確保するために、最新のアップデートの適用を検討する必要があります。さらに、システムのログを確認してエラーの有無を確認するべきです。

  T1 📋 運用担当:
    ネットワーク運用担当者としての観点では、ネットワーク機器のバージョンが最新でない可能性はセキュリティ上のリスクとなり得るため、早期のアップデートを推奨します。さらに、システムのログを定期的に確認してエラーを迅速に発見し、トラブルシューティングを行うことが重要です。エラーの早期発見と対応により、ネットワークの安定性とパフォーマンスを維持することができるため、これらの対策を実施する必要があります。

  🔍 [レビュー] 👁️  レビュー担当:
    分析は適切です。ネットワーク機器のバージョン情報からセキュリティリスクの可能性とアップデートの必要性を指摘し、さらにシステムのログを確認してエラーを早期に発見することで、ネットワークの安定性とパフォーマンスを維持することができるという点は妥当です。設計担当と運用担当それぞれの視点から問題が分析されており、適切な対策が提案されているため、技術的な誤りや見落としは見られません。


📝 コマンド: show bgp summary
----------------------------------------------------------------------
  T1 🔧 設計担当:
    ネットワークの状態を分析すると、BGP（ボーダーゲートウェイ・プロトコル）ライセンスキーが欠けているという警告が表示されており、ネットワーク機器の完全な機能を使用するにはライセンスキーの設定が必要です。また、Peersの状態は「Down peers: 1」で、ペアの一方がダウンしていることがわかります。さらに、Peerの状態は「A